# Sudoku

The game development company "Mind Architect" has published a large collection of Sudoku puzzles on paper and in the form of printed books. These puzzles are written in various handwritings and fonts, and thousands of users solve them every day.

However, recently the product team has faced a serious challenge.

The company's archive consists of tens of thousands of scanned images of Sudoku grids for which no digital version exists. Users want to solve these puzzles in the mobile version of the game, but to do this, the content of each image must first be converted into a numerical grid and then its solution calculated.

The technical team believes the solution lies in combining two skills: **Computer Vision** for accurately reading the numbers from the image, and a **Solving Algorithm** for filling the grid. If even a single number is read incorrectly, the entire puzzle is ruined.

This is where you come in.

## Your Mission

You have been invited to this project as a Computer Vision Engineer. Your mission is to build a system that receives an image of a Sudoku grid, correctly recognizes the numbers inside it, and ultimately returns the complete, solved grid.

## Input Image Sample

Each input is an image of a Sudoku grid, similar to the sample below:

## Dataset Structure

The initial project folder consists of the following two sections:

```
dataset/
├── train/
│   ├── images/       ← Sudoku grid images
│   ├── puzzles/      ← Initial numbers of each grid (CSV)
└── test/
    └── images/       ← Only images (unlabeled)

```

### `train` Folder

This section is for training your model and has two files with the same name (with different extensions) for each puzzle:

| Path | Description |
| --- | --- |
| `train/images/lock_XX.jpg` | Sudoku grid image |
| `train/puzzles/lock_XX.csv` | Numbers present in the image; empty cells are indicated by `0` |

Sample of the `puzzles/lock_01.csv` file (numbers seen in the image):

```
0,8,3,0,0,7,0,6,0
9,1,5,0,0,2,7,4,8
2,0,7,1,0,0,0,0,0
1,9,0,4,0,0,0,3,0
6,5,2,8,0,0,0,7,0
0,0,4,0,0,1,5,0,0
5,0,1,0,4,6,0,9,7
8,4,0,7,0,9,3,0,0
7,2,9,0,0,0,0,0,6

```

You can use this data both to train your number recognition model and to evaluate the reading accuracy of your model by comparing the output with the `puzzles` file.

### `test` Folder

This section only contains images and has no labels. You must read these images yourself and generate the final output for each one.

## Rules and Limitations

* **Prohibition of pre-trained models:** The use of any pre-trained models is strictly prohibited. You must train your model from scratch on the `train` data.
* Solving the grid must be based on standard Sudoku rules: numbers 1 to 9 without repetition in each row, column, and $3 \times 3$ subgrid.

## Submission Method

Your final output must be a compressed file named `submission.zip` with the following structure:

```
submission.zip

├── notebook.ipynb        ← Your complete code notebook

└── submission.csv        ← Final output file

```

* `notebook.ipynb`: The notebook containing all the steps of model training and output generation.
* `submission.csv`: A CSV file where each row is a solved Sudoku grid:
* **First column**: Image file name (e.g., `lock_01`)
* **Columns 2 to 82**: 81 numbers of the solved Sudoku grid, in order from the first cell (top left) to the last cell (bottom right), row by row.



**Sample structure of `submission.csv`:**

```
puzzle_id,1,2,3,4,5,6,7,8,9,…,81

lock_01,4,8,3,9,5,7,2,6,1,9,1,5,3,6,2,7,4,8,…,6

lock_02,7,2,6,4,9,3,8,1,5,3,1,5,7,2,8,9,4,6,…,2

...

```

Each row contains 82 values: one identifier and 81 numbers from the solved 9x9 grid.

## Evaluation Metric

The main metric is **Accuracy** at the level of the entire puzzle. A puzzle is considered "correct" when both of the following conditions are met:

1. **Correct digit recognition:** All numbers present in the image must be read correctly. The numbers in the filled cells in your output must be exactly the same numbers that are in the image. Even a single wrong number invalidates the whole puzzle.
2. **Valid solution:** The output grid must be a valid Sudoku solution that is consistent with the recognized initial numbers.

**About the solutions:** A Sudoku puzzle may have more than one valid solution. Your output does not need to be exactly identical to a reference. **Any different solution is acceptable**, provided that the initial numbers are recognized correctly and the final answer is completely valid according to Sudoku rules.

The final score is calculated as follows:


$$\mathrm{Accuracy} = \frac{\text{Correctly recognized initial digits} \text{ AND } \text{Valid solution (Sudoku rules)}}{\text{Total number of Sudoku puzzles}}$$

$$\mathrm{Score} = 100 \times \mathrm{Accuracy}$$

Since a single incorrect number in the recognition phase invalidates the entire puzzle, the accuracy of your computer vision model is the most important factor in the final score.

**Minimums:** The minimum passing score for this project is set to 30.

During the competition, to prevent overfitting, evaluation is performed only on 30% of the test data, and your temporary score is displayed based on this. At the end of the competition, the submitted answers will be judged on 100% of the test data, and the final score will be obtained from this complete evaluation.

## 📦 Imports & Device Configuration

All standard library, NumPy, OpenCV, and PyTorch imports are collected here. The device is set to GPU if available; CPU is used as a fallback, keeping the notebook runnable in any environment.

In [ ]:
import os
import glob
import random
import csv
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import zipfile
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


/home/norbert/.venvs/ml/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 804: forward compatibility was attempted on non supported HW (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## ⚙️ Section 1 — Dataset Directory Resolution

In [2]:
DATA_DIR = "dataset"
print(f"Dataset directory located at: {DATA_DIR}")

Dataset directory located at: dataset


## 🔍 Section 2 — Grid Detection & Perspective Warping

This section implements the computer vision pipeline to locate and deskew the Sudoku board from a raw photograph:

| Step | OpenCV API | Purpose |
|------|-----------|----------|
| Grayscale read | `cv2.imread(..., IMREAD_GRAYSCALE)` | Remove colour channels |
| Gaussian blur | `cv2.GaussianBlur(img, (7,7), 0)` | Suppress high-frequency noise |
| Adaptive threshold | `cv2.adaptiveThreshold(...)` | Binarise with local contrast |
| Contour detection | `cv2.findContours(...)` | Locate the board outline |
| Convex hull | `cv2.convexHull(c)` | Smooth jagged corner estimates |
| Polygon approx. | `cv2.approxPolyDP(...)` | Reduce hull to 4 corner points |
| Perspective warp | `getPerspectiveTransform` + `warpPerspective` | Rectify board to 450×450 px |

`extract_cells()` then slices the rectified board into 81 individual cell images, applies a 6-pixel margin to remove grid lines, inverts pixel values (so digits are bright on black), and resizes each cell to 28×28 for the classifier.

In [3]:
# -------------------------------------------------------------
# 2. Grid Detection & Perspective Warping
# -------------------------------------------------------------
def order_points(pts):
    pts = pts.reshape(4, 2).astype(np.float32)
    rect = np.zeros((4, 2), dtype=np.float32)
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]
    diff = pts[:, 0] - pts[:, 1]
    rect[1] = pts[np.argmax(diff)]
    rect[3] = pts[np.argmin(diff)]
    return rect

def get_warped(path, size=450):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    blurred = cv2.GaussianBlur(img, (7, 7), 0)
    thresh = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    c = max(contours, key=cv2.contourArea)
    hull = cv2.convexHull(c)
    approx = cv2.approxPolyDP(hull, 0.03 * cv2.arcLength(hull, True), True)
    rect = order_points(approx)
    dst = np.array([[0, 0], [size - 1, 0], [size - 1, size - 1], [0, size - 1]], dtype=np.float32)
    M = cv2.getPerspectiveTransform(rect, dst)
    return cv2.warpPerspective(img, M, (size, size))

def extract_cells(warped, target_size=28, margin=6):
    cells = []
    for r in range(9):
        for c in range(9):
            cell = warped[r * 50 + margin : (r + 1) * 50 - margin, c * 50 + margin : (c + 1) * 50 - margin]
            cell_inv = (255.0 - cell.astype(np.float32)) / 255.0
            res = cv2.resize(cell_inv, (target_size, target_size))
            cells.append(res)
    return np.array(cells, dtype=np.float32)[:, None, :, :]



## 🧠 Section 3 — CNN Architecture (`SudokuCNN`)

`SudokuCNN` classifies each 28×28 cell image into one of **10 classes** (0 = empty, 1–9 = digit).

```
Input  (1×28×28)
  ↓ Conv(1→32, 3×3) + BN + ReLU
  ↓ Conv(32→64, 3×3) + BN + ReLU + MaxPool(2×2)  → 64×14×14
  ↓ Conv(64→128, 3×3) + BN + ReLU + MaxPool(2×2) → 128×7×7
  ↓ Flatten → FC(6272→128) + ReLU + Dropout(0.3)
  ↓ FC(128→10)  [logits]
```

Batch Normalisation after each convolution stabilises training gradients. Dropout before the output head reduces overfitting on the relatively small cell-image dataset.

In [4]:
# -------------------------------------------------------------
# 3. Model Architecture (Trained from scratch)
# -------------------------------------------------------------
class SudokuCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool1 = nn.MaxPool2d(2, 2)
        
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool2 = nn.MaxPool2d(2, 2)
        
        self.fc1 = nn.Linear(128 * 7 * 7, 128)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, 10)
        
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool1(F.relu(self.bn2(self.conv2(x))))
        x = self.pool2(F.relu(self.bn3(self.conv3(x))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x



## 🔄 Section 4 — Data Augmentation

Printed digits in Sudoku puzzles have low intra-class variation, so augmentation is mild but essential for robustness to scan angle, print scale, and lighting.

- **`augment_image()`**: Applies a combined rotation (±6°) + scale (±8%) + translation (±2 px) affine transform using `cv2.getRotationMatrix2D` + `cv2.warpAffine`, followed by random brightness scaling (×0.85–1.15).
- **`AugmentedDataset`**: A PyTorch `Dataset` wrapper that applies augmentation *only to non-empty cells* (`label != 0`). Empty cells are intentionally left unaugmented to preserve the blank-cell appearance the model must distinguish from a digit.

In [5]:
# -------------------------------------------------------------
# 4. Data Augmentation
# -------------------------------------------------------------
def augment_image(img):
    angle = random.uniform(-6, 6)
    scale = random.uniform(0.92, 1.08)
    tx = random.uniform(-2, 2)
    ty = random.uniform(-2, 2)
    M = cv2.getRotationMatrix2D((14, 14), angle, scale)
    M[0, 2] += tx
    M[1, 2] += ty
    aug = cv2.warpAffine(img, M, (28, 28), borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    aug = np.clip(aug * random.uniform(0.85, 1.15), 0, 1)
    return aug

class AugmentedDataset(Dataset):
    def __init__(self, X, y, augment=True):
        self.X = X
        self.y = y
        self.augment = augment
        
    def __len__(self):
        return len(self.X)
        
    def __getitem__(self, idx):
        img = self.X[idx, 0]
        label = self.y[idx]
        if self.augment and label != 0:
            img = augment_image(img)
        return torch.tensor(img[None, :, :], dtype=torch.float32), torch.tensor(label, dtype=torch.int64)



## 🧩 Section 5 — Constraint-Propagation Solver & Guided Decoding

### Solver (`solve_sudoku`)
Uses **Minimum-Remaining-Values (MRV)** backtracking: at each recursive step it picks the empty cell with the *fewest valid candidates*, dramatically pruning the search tree versus naïve left-to-right scan.

### Validity checker (`check_board_valid`)
Verifies the three Sudoku constraints (rows, columns, 3×3 boxes) on the *already-placed* digits without solving — used as a fast guard before launching the expensive backtracker.

### Constraint-guided decoding (`decode_with_solver`)
`decode_with_solver(probs)` bridges CNN uncertainty and logical Sudoku rules:
1. **Greedy decode** (`argmax`) → try to solve directly.
2. **1-substitution search** — for the 25 most uncertain cells, try each runner-up class and re-solve.
3. **2-substitution search** — pairwise correction over the 12 most uncertain cells.
4. **Fallback** — iteratively zero-out the least-confident conflicting clue and re-solve.

In [6]:
# -------------------------------------------------------------
# 5. Sudoku Solver & Constraint-Guided Decoding
# -------------------------------------------------------------
def check_board_valid(grid):
    for r in range(9):
        row = [x for x in grid[r] if x != 0]
        if len(row) != len(set(row)): return False
    for c in range(9):
        col = [grid[r][c] for r in range(9) if grid[r][c] != 0]
        if len(col) != len(set(col)): return False
    for br in range(0, 9, 3):
        for bc in range(0, 9, 3):
            box = [grid[br+i][bc+j] for i in range(3) for j in range(3) if grid[br+i][bc+j] != 0]
            if len(box) != len(set(box)): return False
    return True

def get_candidates(grid, r, c):
    used = set()
    for i in range(9):
        if grid[r][i] != 0: used.add(grid[r][i])
        if grid[i][c] != 0: used.add(grid[i][c])
    br, bc = 3 * (r // 3), 3 * (c // 3)
    for i in range(3):
        for j in range(3):
            if grid[br + i][bc + j] != 0: used.add(grid[br + i][bc + j])
    return [k for k in range(1, 10) if k not in used]

def solve_sudoku(grid):
    min_cands = None
    best_cell = None
    for r in range(9):
        for c in range(9):
            if grid[r][c] == 0:
                cands = get_candidates(grid, r, c)
                if len(cands) == 0:
                    return None
                if min_cands is None or len(cands) < len(min_cands):
                    min_cands = cands
                    best_cell = (r, c)
                    if len(cands) == 1:
                        break
        if min_cands and len(min_cands) == 1:
            break
            
    if best_cell is None:
        return grid
        
    r, c = best_cell
    for k in min_cands:
        grid[r][c] = k
        sol = solve_sudoku(grid)
        if sol is not None:
            return sol
        grid[r][c] = 0
    return None

def decode_with_solver(probs):
    grid = np.argmax(probs, axis=-1)
    if check_board_valid(grid):
        sol = solve_sudoku([list(row) for row in grid])
        if sol is not None:
            return grid.copy(), sol
            
    uncertainties = []
    for r in range(9):
        for c in range(9):
            sorted_idx = np.argsort(probs[r, c])[::-1]
            p1 = probs[r, c, sorted_idx[0]]
            p2 = probs[r, c, sorted_idx[1]]
            uncertainties.append((p1 - p2, r, c, sorted_idx[:4]))
            
    uncertainties.sort(key=lambda x: x[0])
    
    # 1-substitution search
    for gap, r, c, top_cands in uncertainties[:25]:
        for cand in top_cands[1:]:
            cand_grid = [list(row) for row in grid]
            cand_grid[r][c] = cand
            if check_board_valid(cand_grid):
                sol = solve_sudoku([list(row) for row in cand_grid])
                if sol is not None:
                    return np.array(cand_grid), sol
                    
    # 2-substitutions search
    for i in range(min(12, len(uncertainties))):
        _, r1, c1, cands1 = uncertainties[i]
        for cand1 in cands1[1:]:
            for j in range(i + 1, min(12, len(uncertainties))):
                _, r2, c2, cands2 = uncertainties[j]
                for cand2 in cands2[1:]:
                    cand_grid = [list(row) for row in grid]
                    cand_grid[r1][c1] = cand1
                    cand_grid[r2][c2] = cand2
                    if check_board_valid(cand_grid):
                        sol = solve_sudoku([list(row) for row in cand_grid])
                        if sol is not None:
                            return np.array(cand_grid), sol

    # Fallback: remove least certain conflicting clue
    cand_grid = [list(row) for row in grid]
    for gap, r, c, _ in uncertainties:
        if cand_grid[r][c] != 0:
            cand_grid[r][c] = 0
            if check_board_valid(cand_grid):
                sol = solve_sudoku([list(row) for row in cand_grid])
                if sol is not None:
                    return np.array(cand_grid), sol
                    
    return grid.copy(), None



## 🏋️ Section 6 — Training Pipeline

`train_model()` orchestrates a full supervised training run:

- **Optimiser**: AdamW (lr=1e-3, weight_decay=1e-4) — decoupled weight decay prevents L2 regularisation from being absorbed by the adaptive moment scaling.
- **Scheduler**: CosineAnnealingLR anneals the learning rate smoothly to ~0 over `T_max=epochs`, avoiding abrupt LR drops.
- **Loss**: CrossEntropyLoss over 10 digit classes.
- **Ensemble**: Two models are trained with different random seeds (42, 123); their softmax outputs are averaged at inference to reduce variance.

The data loader uses `AugmentedDataset` with `augment=True`, so every training batch benefits from on-the-fly affine augmentation.

In [7]:
# -------------------------------------------------------------
# 6. Training Pipeline
# -------------------------------------------------------------
def train_model(seed, X_train, y_train, epochs=25):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    
    model = SudokuCNN().to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    
    train_ds = AugmentedDataset(X_train, y_train, augment=True)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    
    print(f"Training Model (seed {seed}) for {epochs} epochs...")
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        if epoch % 5 == 0 or epoch == epochs:
            print(f"Epoch {epoch:2d}/{epochs}: loss = {total_loss/len(train_loader):.4f}")
            
    model.eval()
    return model

# Load training data
train_imgs = sorted(glob.glob(os.path.join(DATA_DIR, "train", "images", "*.jpg")))
print(f"Loading {len(train_imgs)} train images...")

X_train_list, y_train_list = [], []
for p in train_imgs:
    csv_p = p.replace("images", "puzzles").replace(".jpg", ".csv")
    with open(csv_p) as f:
        grid = [int(x) for line in f if line.strip() for x in line.strip().split(",")]
    cells = extract_cells(get_warped(p))
    X_train_list.append(cells)
    y_train_list.append(grid)
    
X_train = np.concatenate(X_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)
print(f"Extracted training dataset: X={X_train.shape}, y={y_train.shape}")

# Train ensemble of models
seeds = [42, 123]
models = []
for s in seeds:
    m = train_model(s, X_train, y_train, epochs=25)
    models.append(m)

Loading 100 train images...
Extracted training dataset: X=(8100, 1, 28, 28), y=(8100,)
Training Model (seed 42) for 25 epochs...
Epoch  5/25: loss = 0.0932
Epoch 10/25: loss = 0.0384
Epoch 15/25: loss = 0.0152
Epoch 20/25: loss = 0.0093
Epoch 25/25: loss = 0.0073
Training Model (seed 123) for 25 epochs...
Epoch  5/25: loss = 0.1894
Epoch 10/25: loss = 0.0833
Epoch 15/25: loss = 0.0397
Epoch 20/25: loss = 0.0220
Epoch 25/25: loss = 0.0235


## 📤 Section 7 — Test Inference & Submission Generation

This final section runs the full end-to-end pipeline on the held-out test images:
1. **Warp** each image to a canonical 450×450 board.
2. **Extract** 81 cells and batch-infer with both ensemble models.
3. **Average** softmax probabilities across models.
4. **Decode** with `decode_with_solver()` using constraint propagation.
5. **Fallback**: if the solver cannot find a valid solution, fill remaining zeros with 1 (a safe placeholder).
6. Write `submission.csv` with columns `puzzle_id, 1…81`.

In [ ]:
# -------------------------------------------------------------
# 7. Generate Submission for Test Set
# -------------------------------------------------------------
test_imgs = sorted(glob.glob(os.path.join(DATA_DIR, "test", "images", "*.jpg")))
print(f"Processing {len(test_imgs)} test images for submission...")

submission_rows = []
header = ["puzzle_id"] + [str(i) for i in range(1, 82)]
submission_rows.append(header)

solved_count = 0
for p in test_imgs:
    pid = os.path.splitext(os.path.basename(p))[0]
    warped = get_warped(p)
    cells = extract_cells(warped)
    
    with torch.no_grad():
        probs = torch.zeros(81, 10, device=device)
        cell_tensor = torch.tensor(cells, dtype=torch.float32, device=device)
        for m in models:
            probs += F.softmax(m(cell_tensor), dim=-1)
        probs /= len(models)
        probs = probs.cpu().numpy().reshape(9, 9, 10)
        
    initial_grid, solved_grid = decode_with_solver(probs)
    if solved_grid is not None:
        solved_count += 1
        flat_solution = [int(x) for row in solved_grid for x in row]
    else:
        print(f"Warning: Fallback solution used for {pid}")
        flat_solution = [int(x) if x != 0 else 1 for row in initial_grid for x in row]
        
    row = [pid] + [str(x) for x in flat_solution]
    submission_rows.append(row)

print(f"Successfully solved {solved_count}/{len(test_imgs)} test puzzles!")

# Write submission.csv
csv_path = "submission.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(submission_rows)

print(f"Wrote {len(submission_rows)} rows to {csv_path}")

Processing 100 test images for submission...
Successfully solved 100/100 test puzzles!
Wrote 101 rows to submission.csv


In [ ]:
if not os.path.exists(os.path.join(os.getcwd(), 'notebook.ipynb')):
    %notebook -e notebook.ipynb

def create_submission_zip(zip_name="submission.zip", notebook_name="notebook.ipynb", sub_dir="."):
    print(f"Creating {zip_name} ...")
    compression = zipfile.ZIP_DEFLATED
    
    with zipfile.ZipFile(zip_name, mode="w", compression=compression) as zf:
        if os.path.exists(notebook_name):
            zf.write(notebook_name, arcname=notebook_name)
            print(f"Added: {notebook_name}")
        else:
            raise FileNotFoundError(f"Error: '{notebook_name}' not found! Process aborted.")

        csv_path = os.path.join(sub_dir, 'submission.csv')
        
        if os.path.exists(csv_path) and os.path.isfile(csv_path):
            zf.write(csv_path, arcname='submission.csv')
            print(f"Added: {csv_path}")
        else:
            raise FileNotFoundError(f"Error: 'submission.csv' not found exactly in '{sub_dir}'! Process aborted.")
            
    print(f"\nSuccessfully created {zip_name}")

create_submission_zip()